In [ ]:
# ─────────────────────────────────────────────
# 📦 Core Libraries
# ─────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────
# 📊 Data Visualization
# ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# ⚙️ Preprocessing
# ─────────────────────────────────────────────
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    LabelEncoder,
    OneHotEncoder
)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    StratifiedKFold
)

# ─────────────────────────────────────────────
# 🤖 Classification Models
# ─────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

# ─────────────────────────────────────────────
# 📈 Evaluation Metrics
# ─────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

# ─────────────────────────────────────────────
# 🔧 Utilities
# ─────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print('✅ All libraries imported successfully!')

In [ ]:
# ─────────────────────────────────────────────
# 📂 STEP 1 — Data Loading
# ─────────────────────────────────────────────

# Load the dataset
df = pd.read_csv('Iris.csv')

print('=== First 5 rows of the dataset ===')
display(df.head())
print('\nDataset shape:', df.shape)

In [ ]:
# ─────────────────────────────────────────────
# 📊 STEP 2 — Visualizations
# ─────────────────────────────────────────────

features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
sns.set_theme(style='darkgrid', palette='Set2')

# ── 4.1  Feature Distributions (Histograms) ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('🌿 Iris — Feature Distributions by Species', fontsize=16, fontweight='bold')
for ax, feature in zip(axes.flatten(), features):
    for species in df['Species'].unique():
        ax.hist(df[df['Species'] == species][feature], alpha=0.6, label=species, bins=15)
    ax.set_title(feature)
    ax.set_xlabel('Value (cm)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# ── 4.2  Correlation Heatmap ──
plt.figure(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('🔗 Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 4.3  Pairplot ──
sns.pairplot(df[features + ['Species']], hue='Species', diag_kind='kde', height=2.2)
plt.suptitle('📌 Pairplot — All Features', y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# 🧹 STEP 3 — Preprocessing
# ─────────────────────────────────────────────

# Drop the Id column (not a feature)
df.drop(columns=['Id'], inplace=True)

# ── Features and Target ──
X = df.drop(columns=['Species'])
y = df['Species']

# ── Encode target labels: Iris-setosa=0, Iris-versicolor=1, Iris-virginica=2 ──
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print('Classes      :', le.classes_)
print('Encoded vals :', list(set(y_encoded)))

# ── Feature Scaling (StandardScaler: mean=0, std=1) ──
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print('\n=== Scaled Features — First 5 rows ===')
display(X_scaled.head())
print('\n✅ Preprocessing complete!')
print(f'   Features shape : {X_scaled.shape}')
print(f'   Target shape   : {y_encoded.shape}')

In [ ]:
# ─────────────────────────────────────────────
# 🤖 STEP 4 — Model Training & Saving
# ─────────────────────────────────────────────
import pickle

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate the model
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Model Accuracy: {accuracy * 100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Save the model to a pickle file
with open('iris_rf_model.pkl', 'wb') as file:
    pickle.dump(rf_model, file)

print('✅ Model successfully trained and saved as iris_rf_model.pkl')